In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.utils.class_weight import compute_class_weight

file_name = 'matched_files_fixed.csv' 
label_column_name = 'label_activity' 
try:
    print(f"--- Loading Data from '{file_name}' ---")
    df = pd.read_csv(file_name)
    print("File loaded successfully!")
    print(f"Total rows loaded: {len(df)}")
    if label_column_name in df.columns:
        print(f"\nFound the label column: '{label_column_name}'")
        print("Step 1: Class Distribution (All Labels)")
        class_counts_all = df[label_column_name].value_counts()
        print("\n Class Counts (Number of samples per class - INCLUDING unknown):")
        print(class_counts_all)
        
        class_percentages_all = df[label_column_name].value_counts(normalize=True) * 100
        print("\n Class Percentages (%):")
        print(class_percentages_all.round(2))
        
        print("\n Visual Distribution:")
        for label, count in class_counts_all.items():
            percentage = class_percentages_all[label]
            bar = '█' * int(percentage)
            print(f"{label:15s} | {bar} {percentage:.2f}% ({count} samples)")
        
        print(f"\n--- Filtering Data for Class Weights ---")
        print(f"Total rows: {len(df)}")
        df_filtered = df[df[label_column_name] != 'unknown']
        print(f"Matched rows (excluding 'unknown'): {len(df_filtered)}")
        print(f"Unknown rows: {len(df) - len(df_filtered)}")
        
        if len(df_filtered) == 0:
            print("\n Error: No valid labels found after filtering 'unknown' values.")
            print("Please check your data.")
        else:
            print("\n Class Counts (excluding 'unknown'):")
            print(Class_Counts)
            
            class_percentages = df_filtered[label_column_name].value_counts(normalize=True) * 100
            print("\n Class Percentages (%):")
            print(class_percentages.round(2))
            
            print("\n Visual Distribution:")
            for label, count in class_counts.items():
                percentage = class_percentages[label]
                bar = '█' * int(percentage)
                print(f"{label:15s} | {bar} {percentage:.2f}% ({count} samples)")

            print("Step 2: Calculated Class Weights (Matched Labels)")
            
            labels = df_filtered[label_column_name].to_numpy()
            class_labels = np.unique(labels)
            
            class_weights = compute_class_weight(
                class_weight='balanced',
                classes=class_labels,
                y=labels
            )
            
            class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)
            print("\n  Class Weights (for matched labels - handling imbalance):")
            for i, label in enumerate(class_labels):
                print(f"  Class: '{label}' -> Weight: {class_weights[i]:.4f}")
                
            print(f"\n PyTorch Tensor of weights:")
            print(f"   {class_weights_tensor}")
            
            class_weight_dict = {label: weight for label, weight in zip(class_labels, class_weights)}
            print(f"\n Class Weight Dictionary:")
            print(f"   {class_weight_dict}")
            
            print("Step 3: Calculated Class Weights (ALL Labels Including Unknown)")
            labels_all = df[label_column_name].to_numpy()
            class_labels_all = np.unique(labels_all)
            
            class_weights_all = compute_class_weight(
                class_weight='balanced',
                classes=class_labels_all,
                y=labels_all
            )
            
            class_weights_all_tensor = torch.tensor(class_weights_all, dtype=torch.float)
            
            print("\n  Class Weights (for ALL labels including 'unknown'):")
            for i, label in enumerate(class_labels_all):
                print(f"  Class: '{label}' -> Weight: {class_weights_all[i]:.4f}")
                
            print(f"\n PyTorch Tensor of weights:")
            print(f"   {class_weights_all_tensor}")
            
            class_weight_dict_all = {label: weight for label, weight in zip(class_labels_all, class_weights_all)}
            print(f"\n Class Weight Dictionary (ALL):")
            print(f"   {class_weight_dict_all}")
            
            summary_all_df = pd.DataFrame({
                'class': class_counts_all.index,
                'count': class_counts_all.values,
                'percentage': class_percentages_all.values,
                'weight': [class_weight_dict_all[c] for c in class_counts_all.index]
            })
            
            summary_all_df.to_csv('class_distribution_all.csv', index=False)
            print(" Full distribution (including unknown) saved to: class_distribution_all.csv")
            
            summary_df = pd.DataFrame({
                'class': class_counts.index,
                'count': class_counts.values,
                'percentage': class_percentages.values,
                'weight': [class_weight_dict[c] for c in class_counts.index]
            })
            
            summary_df.to_csv('class_distribution_matched.csv', index=False)
            print(" Matched distribution summary saved to: class_distribution_matched.csv")
            
            np.save('class_weights_matched.npy', class_weights)
            print(" Matched class weights saved to: class_weights_matched.npy")
            
            np.save('class_weights_all.npy', class_weights_all)
            print(" All class weights saved to: class_weights_all.npy")
            
            import json
            with open('class_weights_matched.json', 'w') as f:
                json.dump({str(k): float(v) for k, v in class_weight_dict.items()}, f, indent=2)
            print(" Matched class weight dictionary saved to: class_weights_matched.json")
            
            with open('class_weights_all.json', 'w') as f:
                json.dump({str(k): float(v) for k, v in class_weight_dict_all.items()}, f, indent=2)
            print(" All class weight dictionary saved to: class_weights_all.json")
            
    else:
        print(f"\n Error: Column '{label_column_name}' not found in the CSV file.")
        print(f"Available columns: {list(df.columns)}")
        print(f"\nPlease update the 'label_column_name' variable to one of the above columns.")

except FileNotFoundError:
    print(f" Error: The file '{file_name}' was not found.")
    print(f"Please make sure the file is in the current directory.")
except Exception as e:
    print(f" An error occurred: {e}")
    import traceback
    traceback.print_exc()

--- Loading Data from 'matched_files_fixed.csv' ---
File loaded successfully!
Total rows loaded: 287

Found the label column: 'label_activity'
Step 1: Class Distribution (All Labels)

 Class Counts (Number of samples per class - INCLUDING unknown):
label_activity
unknown    145
NIGHT       86
ORD         39
PLAY         9
FFR          8
Name: count, dtype: int64

 Class Percentages (%):
label_activity
unknown    50.52
NIGHT      29.97
ORD        13.59
PLAY        3.14
FFR         2.79
Name: proportion, dtype: float64

 Visual Distribution:
unknown         | ██████████████████████████████████████████████████ 50.52% (145 samples)
NIGHT           | █████████████████████████████ 29.97% (86 samples)
ORD             | █████████████ 13.59% (39 samples)
PLAY            | ███ 3.14% (9 samples)
FFR             | ██ 2.79% (8 samples)

--- Filtering Data for Class Weights ---
Total rows: 287
Matched rows (excluding 'unknown'): 142
Unknown rows: 145

 Class Counts (excluding 'unknown'):
 An error o

Traceback (most recent call last):
  File "C:\Users\ukart\AppData\Local\Temp\ipykernel_26612\1317708437.py", line 45, in <module>
    print(class_counts)
          ^^^^^^^^^^^^
NameError: name 'class_counts' is not defined. Did you mean: 'class_counts_all'?
